In [2]:
!pip install langchain langchain-core langchain-community langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.6/79.6 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.5/571.5 kB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 53.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.7 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.6.0
    Uninstalling langchain-core-1.6.0:
      Successfully uninstalled langchain-core-1.6.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-co

In [3]:
import numpy as np #these are used for data analysis purposes
import pandas as pd

In [4]:
data=pd.read_excel("/content/My_Financial_Report_Dataset.xlsx")

In [5]:
data.head()

,Quarter,Business_Unit,Revenue_MUSD,COGS_MUSD,Operating_Expense_MUSD,EBITDA_MUSD,Net_Profit_MUSD,YoY_Growth_%,Market_Share_%,Customer_Churn_%,CapEx_MUSD,Region,Risk_Flag,Notes
0,Q1-2025,Payments,431,167.5,113.2,141.3,84.9,10.1,12.9,8.6,48.8,North America,Low,Cloud migration investment
1,Q2-2025,Corporate Banking,711,354.2,159.4,199.3,144.6,1.3,14.0,8.7,105.3,Europe,High,AI automation reduced servicing cost
2,Q1-2025,Wealth,319,178.9,68.8,67.7,7.7,15.0,12.4,6.0,93.9,Europe,Low,Higher regulatory compliance expense
3,Q2-2025,Payments,623,285.2,165.4,170.3,104.9,14.4,8.4,4.8,105.0,APAC,Medium,Cloud migration investment
4,Q2-2025,Wealth,235,93.7,55.1,93.2,45.1,-0.6,16.7,8.1,113.4,Europe,Low,Cloud migration investment


In [7]:
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import LLMChain, SequentialChain
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI

In [9]:
import os
os.environ["GEMINI_API_KEY"]="API"

In [10]:
llm=ChatGoogleGenerativeAI(
    model='gemini-3.6-flash'
)

Chain 1 - Responsible for extracting KPIs from the financial data

In [11]:
#Prompt for Chain 1
kpi_prompt=PromptTemplate(
    input_variables=["data"],
    template="""
    You are a Financial Analyst and you have been provided with financial data of a Company.
    Extract the most important financial and business KPIs from the data in bullet points format

    Do not provide any recommendations and not even a summary.

    financial data:{data}"""
)

In [12]:
#Creating Chain 1
chain_1=kpi_prompt | llm | StrOutputParser()

In [13]:
#Executing Chain 1
kpis=chain_1.invoke({"data":data})

In [14]:
print(kpis)

* **Total Revenue:** $35,468.00 Million USD
* **Total Cost of Goods Sold (COGS):** $16,339.70 Million USD
* **Total Gross Profit:** $19,128.30 Million USD
* **Gross Profit Margin:** 53.93%
* **Total Operating Expenses:** $8,662.90 Million USD
* **Total EBITDA:** $9,642.80 Million USD
* **EBITDA Margin:** 27.19%
* **Total Net Profit:** $6,367.60 Million USD
* **Net Profit Margin:** 17.95%
* **Total Capital Expenditure (CapEx):** $3,741.00 Million USD
* **Average YoY Revenue Growth Rate:** 6.91%
* **Average Market Share:** 14.28%
* **Average Customer Churn Rate:** 5.75%

* **Revenue by Business Unit:**
  * **Corporate Banking:** $15,595.00 Million USD (43.97% of total)
  * **Payments:** $10,147.00 Million USD (28.61% of total)
  * **Insurance:** $4,884.00 Million USD (13.77% of total)
  * **Wealth:** $4,842.00 Million USD (13.65% of total)
  * **Retail Banking:** $2,782.00 Million USD (7.84% of total)

* **Revenue by Region:**
  * **APAC:** $11,595.00 Million USD (32.69% of total)
  * **

Chain 2 - Receives KPIs from Chain 1 and creates performance benchmarks based on the KPIs received

In [15]:
#prompt for Chain 2
benchmark_prompt=PromptTemplate(
    input_variables=['kpis'],
    template="""
    You are a Senior Strategy Consultant and your job is to critically analyze
    the KPIs provided and establish appropriate performance benchmarks for Management Evaluation

    For every major KPI:
    - State the current performance
    - Define reasonable target/benchmarks
    - Classify the performance as:
      - Strong
      - Moderate
      - Needs improvement

    Focus on:
    - Revenue growth
    - EBITDA margin
    - Net profit margin

    Return the benchmarks in a structured format

    kpis:{kpis}"""
)

In [16]:
#Creating Chain 2
chain_2=benchmark_prompt | llm | StrOutputParser()

In [22]:
#Executing Chain 2
benchmarks=chain_2.invoke({"kpis":kpis})

In [ ]:
print(benchmarks)

Chain 3 - Creates an Executive Report based on the performance benchmarks received from Chain 2

In [19]:
#prompt for Chain 3
executive_prompt=PromptTemplate(
    input_variables=['benchmarks'],
    template="""
    You are a Senior Strategy Consultant and you are supposed to read and understand
    the performance benchmarks of this company and create an executive report using the benchmarks as reference

    benchmarks:{benchmarks}
    """)

In [20]:
#Creating Chain 3
chain_3=executive_prompt | llm | StrOutputParser()

In [23]:
#Executing Chain 3
executive_report=chain_3.invoke({"benchmarks":benchmarks})

Activity: AI Content Refinement Pipeline

Building a 3 step Sequential LangChain workflow where:
- First Step - Generates content
- Second Step - Improves the content
- Third Step - Social Media Post using the refined content